# Create the final dataset

This notebook joins the three task files into one final dataset.

| Input file | Columns |
|---|---|
| `split_real_fake.csv` | `image_id`, `label` |
| `split_gender.csv` | `image_id`, `gender` |
| `split_age.csv` | `image_id`, `age` |

The files are joined on `image_id`. The final dataset has these columns:

| image_id | label | gender | age_group |
|---|---|---|---|
| 1 | real | male | 18-25 |
| 2 | real | male | 26-35 |
| 3 | fake | unknown | 36-50 |
| 4 | fake | unknown | 50+ |
| 5 | real | female | not_applicable |

The output file is `final_image_dataset.csv`, saved in the `data` folder.
The existing `final_dataset.csv` in your data folder is not touched.

Run `07_combine_split_csv_files.ipynb` first, because it creates the three input files.

## Step 1: Set the paths and options

In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data")

REAL_FAKE_FILE = DATA_DIR / "split_real_fake.csv"
GENDER_FILE    = DATA_DIR / "split_gender.csv"
AGE_FILE       = DATA_DIR / "split_age.csv"
OUTPUT_FILE    = DATA_DIR / "final_image_dataset.csv"

# Some images may be in only one input file (for example under_18 images that were added by hand
# only to the age folder). Those images have empty values in the other columns.
DROP_INCOMPLETE_ROWS = False       # True = remove the rows that have an empty value

for file in (REAL_FAKE_FILE, GENDER_FILE, AGE_FILE):
    if not file.exists():
        raise FileNotFoundError("File not found: " + str(file) + ". Run notebook 07 first.")

## Step 2: Read the three files

Everything is read as text, so ids such as `007` stay unchanged.
If an `image_id` appears more than once in a file, a warning is printed and only the first row is kept.

In [2]:
def read_file(file_path):
    df = pd.read_csv(file_path, dtype=str)
    repeated = df["image_id"].duplicated().sum()
    print(f"{file_path.name:22s} rows = {len(df):5d} | repeated image_id = {repeated}")
    if repeated > 0:
        print("   WARNING: only the first row of each repeated image_id is kept")
        df = df.drop_duplicates("image_id")
    return df

real_fake_df = read_file(REAL_FAKE_FILE)
gender_df    = read_file(GENDER_FILE)
age_df       = read_file(AGE_FILE).rename(columns={"age": "age_group"})   # the final column is named age_group

split_real_fake.csv    rows =  5088 | repeated image_id = 0
split_gender.csv       rows =  5088 | repeated image_id = 0
split_age.csv          rows =  5088 | repeated image_id = 0


## Step 3: Join the files

An outer join keeps every image that appears in at least one file.

In [3]:
final = real_fake_df.merge(gender_df, on="image_id", how="outer").merge(age_df, on="image_id", how="outer")

# sort by image id (numbers first, then text ids)
final["sort_key"] = pd.to_numeric(final["image_id"], errors="coerce")
final = final.sort_values(["sort_key", "image_id"]).drop(columns="sort_key").reset_index(drop=True)
final = final[["image_id", "label", "gender", "age_group"]]

incomplete = final[final.isna().any(axis=1)]
print("Rows in the final dataset :", len(final))
print("Rows with an empty value  :", len(incomplete))

if len(incomplete) > 0:
    print("\nThese images are missing in one or more input files (first 10):")
    print(incomplete.head(10).to_string(index=False))

Rows in the final dataset : 5088
Rows with an empty value  : 0


## Step 4: Check the values

In [4]:
expected = {
    "label":     ["real", "fake"],
    "gender":    ["female", "male", "unknown"],
    "age_group": ["18-25", "26-35", "36-50", "50+", "not_applicable", "under_18"],
}

for column, allowed in expected.items():
    unexpected = sorted(set(final[column].dropna()) - set(allowed))
    if unexpected:
        print(f"WARNING: unexpected values in '{column}':", unexpected)
    print("=====", column, "=====")
    print(final[column].value_counts(dropna=False), "\n")

===== label =====
label
real    2593
fake    2495
Name: count, dtype: int64 

===== gender =====
gender
unknown    2612
female     1453
male       1023
Name: count, dtype: int64 

===== age_group =====
age_group
not_applicable    2638
26-35             1484
36-50              338
18-25              330
50+                219
under_18            79
Name: count, dtype: int64 



## Step 5: Save the final dataset

In [5]:
if DROP_INCOMPLETE_ROWS:
    final = final.dropna().reset_index(drop=True)
    print("Rows with empty values were removed.")

final.to_csv(OUTPUT_FILE, index=False)
print("Saved:", OUTPUT_FILE.name, "| rows =", len(final))
final.head(10)

Saved: final_image_dataset.csv | rows = 5088


,image_id,label,gender,age_group
0,1,real,male,26-35
1,2,real,male,under_18
2,3,real,male,26-35
3,4,real,male,26-35
4,5,real,male,36-50
5,6,real,male,18-25
6,7,real,female,50+
7,8,real,female,18-25
8,9,real,male,36-50
9,10,real,female,not_applicable
